In [1]:
import requests 
import zipfile
from pathlib import Path

def load_metadata(url: str, name: str, out: str = "/home/hmack/Development/smartrodent_experiments/datasets/",  ): 

    dest = Path(out) / name

    zip_path = (dest).with_suffix(".zip")

    dest.mkdir(parents=True, exist_ok=True)

    with requests.get(url, stream = True) as r: 
        r.raise_for_status() 
        with open(zip_path, "wb") as f: 
            for chunk in r.iter_content(chunk_size=1024*1024): 
                f.write(chunk)


    with zipfile.ZipFile(zip_path) as z: 
        z.extractall(dest)

## download metadata for AMMonitor Camera Traps


In [ ]:
url = "https://lilawildlife.blob.core.windows.net/lila-wildlife/ammonitor-camera-traps/ammonitor-camera-traps.zip"
name = "ammonitor"

dest = "/home/hmack/Development/smartrodent_experiments/datasets"
load_metadata(url, name, out=dest)

In [14]:
import json
import pandas as pd 

with open("/home/hmack/Development/smartrodent_experiments/datasets/ammonitor-camera-traps.json") as f:
    d = json.load(f)          # ~10-15 GB peak RAM; you have 42 GB free

images = pd.DataFrame(d["images"])        # id, file_name, location, datetime, dataset, seq_id, frame_num, seq_num_frames
annots = pd.DataFrame(d["annotations"])   # id, image_id, category_id
cats   = pd.DataFrame(d["categories"])    # id, name

df = annots.merge(images, left_on="image_id", right_on="id", suffixes=("_ann", "_img")).merge(cats.rename(columns={"id": "category_id", "name": "category"}), on="category_id")


In [39]:
sorted(df.category.unique())

['accipiter sp.',
 'american crow',
 'american marten',
 'american robin',
 'american woodcock',
 'animal sp.',
 'barred owl',
 'beaver',
 'bird sp.',
 'black bear',
 'black-backed woodpecker',
 'black-capped chickadee',
 'blue jay',
 'bobcat',
 'broad-winged hawk',
 'canada jay',
 'canada lynx',
 'canid sp.',
 'chickadee sp.',
 'chipmunk',
 'common raven',
 'corvid sp.',
 'cottontail rabbit',
 'coyote',
 'dark-eyed junco',
 'domestic cat',
 'domestic dog',
 'empty',
 'felid sp.',
 'fisher',
 'flying squirrel sp.',
 'gray catbird',
 'gray fox',
 'gray squirrel',
 'great horned owl',
 'grouse sp.',
 'hairy woodpecker',
 'hermit thrush',
 'horse',
 'insect sp.',
 'long-tailed weasel',
 'mammal sp.',
 'mink',
 'moose',
 'mouse sp.',
 'muskrat',
 'northern flicker',
 'northern goshawk',
 'northern saw-whet owl',
 'opossum',
 'owl sp.',
 'pileated woodpecker',
 'porcupine',
 'rabbit or hare sp.',
 'raccoon',
 'red fox',
 'red squirrel',
 'red-breasted nuthatch',
 'red-tailed hawk',
 'river 

In [28]:
list(filter(lambda x: any(f in x.lower() for f in ["mus", "mouse", "rat", "rattus", "domestic", "cat", "dog", "rodent", "murid", "shrew", "vole"]),  df.category.unique()))

['domestic dog',
 'bobcat',
 'mouse sp.',
 'muskrat',
 'domestic cat',
 'gray catbird']

In [ ]:
relevant_categories = [
    "empty",  
    "domestic cat", 
    "domestic dog", 
    "mouse sp.", 
]

df_filtered = df.loc[df["category"].str.strip().str.lower().isin(relevant_categories), :]

In [41]:
for cat in relevant_categories: 
    df_filtered = df.loc[df["category"].str.strip().str.lower().isin([cat,]), :]
    print(cat, ", ",   len(df_filtered))

empty ,  615270
domestic cat ,  34
domestic dog ,  1918
mouse sp. ,  7307
felid sp. ,  14


In [37]:
len(df_filtered)

624529

## download metadata for felidae-conservation-fund california dataset

In [ ]:
url = "https://lilawildlife.blob.core.windows.net/lila-wildlife/felidae-conservation-fund/felidae_conservation_fund_2020_2025.zip"
name = "felidae_conservation"
dest = "/home/hmack/Development/smartrodent_experiments/datasets"
load_metadata(url, name, out=dest)

In [1]:
import json
import pandas as pd 
metadata_path = "/home/hmack/Development/smartrodent_experiments/datasets/felidae_conservation/felidae_conservation_fund_2020_2025.json"

with open(metadata_path) as f:
    d = json.load(f)          # ~10-15 GB peak RAM; you have 42 GB free

images = pd.DataFrame(d["images"])        # id, file_name, location, datetime, dataset, seq_id, frame_num, seq_num_frames
annots = pd.DataFrame(d["annotations"])   # id, image_id, category_id
cats   = pd.DataFrame(d["categories"])    # id, name

df = annots.merge(images, left_on="image_id", right_on="id", suffixes=("_ann", "_img")).merge(cats.rename(columns={"id": "category_id", "name": "category"}), on="category_id")

sorted(df.category.unique())

['acorn woodpecker',
 'american badger',
 'american crow',
 'american robin',
 'barn owl',
 'bat',
 'black bear',
 'black-tailed jackrabbit',
 'bobcat',
 'brush rabbit',
 'burrowing owl',
 'california quail',
 'california thrasher',
 'california towhee',
 'canada goose',
 'cattle',
 'common raven',
 'cottontail rabbit',
 'coyote',
 'dark-eyed junco',
 'domestic cat',
 'domestic dog',
 'domestic horse',
 'duck species',
 'eastern grey squirrel',
 'elk or deer',
 'gray fox',
 'great horned owl',
 'heron or egret',
 'invertebrate',
 "merriam's chipmunk",
 'mourning dove',
 'mouse or rat',
 'mule deer',
 'northern band-tailed pigeon',
 'northern flicker',
 'prey-bird',
 'prey-mammal',
 'prey-unknown',
 'puma',
 'rabbit',
 'raccoon',
 'red fox',
 'red-shouldered hawk',
 'red-tailed hawk',
 'reptile',
 'river otter',
 'spotted skunk',
 'spotted towhee',
 "steller's jay",
 'striped skunk',
 'tule elk',
 'turkey',
 'turkey vulture',
 'unknown',
 'unknown bird',
 'unknown hawk',
 'unknown night

In [5]:
relevant_categories = list(filter(lambda x: any(f in x.lower() for f in ["rat", "mouse", "vole", "shrew", "mus", "rattus", "murid", "rodent"]), df.category.unique()))
relevant_categories

['mouse or rat', 'invertebrate']

this dataset at least provides a 'mouse or rat' label, so it can be useful for the detection level

In [6]:
for cat in relevant_categories: 
    df_filtered = df.loc[df["category"].str.strip().str.lower().isin([cat,]), :]
    print(cat, ", ",   len(df_filtered))

mouse or rat ,  8700
invertebrate ,  90


# download metadata for wsu-lynx dataset 

In [47]:
url = "https://lilawildlife.blob.core.windows.net/lila-wildlife/wsu-lynx/wsu-lynx.26.02.13.1705.zip"
name = "wsu_lynx"
dest = "/home/hmack/Development/smartrodent_experiments/datasets"
load_metadata(url, name, out=dest)

In [7]:
import json
import pandas as pd 
metadata_path = "/home/hmack/Development/smartrodent_experiments/datasets/wsu_lynx/wsu-lynx.json"

with open(metadata_path) as f:
    d = json.load(f)          # ~10-15 GB peak RAM; you have 42 GB free

images = pd.DataFrame(d["images"])        # id, file_name, location, datetime, dataset, seq_id, frame_num, seq_num_frames
annots = pd.DataFrame(d["annotations"])   # id, image_id, category_id
cats   = pd.DataFrame(d["categories"])    # id, name

df = annots.merge(images, left_on="image_id", right_on="id", suffixes=("_ann", "_img")).merge(cats.rename(columns={"id": "category_id", "name": "category"}), on="category_id")


sorted(df.category.unique())

['alces alces',
 'aves',
 'canis latrans',
 'canis lupus',
 'cervus canadensis',
 'domestic cattle',
 'domestic horse',
 'empty',
 'erethizon dorsatum',
 'lepus americanus',
 'lynx canadensis',
 'lynx rufus',
 'marmota caligata',
 'marmota flaviventris',
 'martes americana',
 'mephitis mephitis',
 'odocoileus hemionus',
 'odocoileus virginianus',
 'oreamnos americanus',
 'ovis canadensis',
 'procyon lotor',
 'puma concolor',
 'small mammal',
 'taxidea taxus',
 'unknown',
 'ursus americanus']

In [65]:
list(filter(lambda x: any(f in x.lower() for f in ["rat", "mouse", "vole", "shrew", "mus", "rattus", "murid", "rodent"]), df.category.unique()))

[]

# download metadata for california small mammals dataset 

In [8]:
url = "https://lilawildlife.blob.core.windows.net/lila-wildlife/california-small-animals/california_small_animals_with_sequences.zip"
name = "california_small_mammals"
dest = "/home/hmack/Development/smartrodent_experiments/datasets"
load_metadata(url, name, out=dest)

NameError: name 'load_metadata' is not defined

In [9]:
import json
import pandas as pd 
metadata_path = "/home/hmack/Development/smartrodent_experiments/datasets/california_small_mammals/california_small_animals_with_sequences.json"

with open(metadata_path) as f:
    d = json.load(f)          # ~10-15 GB peak RAM; you have 42 GB free

images = pd.DataFrame(d["images"])        # id, file_name, location, datetime, dataset, seq_id, frame_num, seq_num_frames
annots = pd.DataFrame(d["annotations"])   # id, image_id, category_id
cats   = pd.DataFrame(d["categories"])    # id, name

df = annots.merge(images, left_on="image_id", right_on="id", suffixes=("_ann", "_img")).merge(cats.rename(columns={"id": "category_id", "name": "category"}), on="category_id")


sorted(df.category.unique())

['alameda striped racer',
 'american black bear',
 'american bullfrog',
 'american mink',
 'ammospermophilus species',
 'amphibian',
 'anaxyrus species',
 'animal',
 'arachnids',
 'arboreal salamander',
 'arvicolinae subfamily',
 'ash-throated flycatcher',
 'aspidoscelis species',
 'baja california coachwhip',
 "belding's ground squirrel",
 "belding's orange-throated whiptail",
 "bewick's wren",
 'bird',
 'black-tailed jackrabbit',
 "blainville's horned lizard",
 'blank',
 'bobcat',
 "botta's pocket gopher",
 "brewer's blackbird",
 'broad-footed mole',
 'brown rat',
 'brush rabbit',
 "bullock's oriole",
 'bumblebees',
 'bushy-tailed woodrat',
 'butterflies and moths',
 'cactus mouse',
 'california ground squirrel',
 'california kangaroo rat',
 'california kingsnake',
 'california quail',
 'california red-legged frog',
 'california red-sided gartersnake',
 'california scrub-jay',
 'california thrasher',
 'california towhee',
 'california vole',
 'canine family',
 'canyon wren',
 'centip

In [12]:
relevant_categories = sorted(list(filter(lambda x: any(f.lower() in x.lower() for f in ["rat", "mouse", "vole", "shrew", "murid", "rodent", "sorex", "crocidura", "apodemus", "Myodes", "Arvicol", "mus", "rattus"]), df.category.unique())))
relevant_categories

['arvicolinae subfamily',
 'brown rat',
 'bushy-tailed woodrat',
 'cactus mouse',
 'california kangaroo rat',
 'california vole',
 'desert woodrat',
 'dusky-footed woodrat',
 'house mouse',
 'house rat',
 'mojave rattlesnake',
 'mouse species',
 'muridae family',
 'muskrat',
 'mustelinae subfamily',
 'north american deermouse',
 'northern grasshopper mouse',
 'northern pacific rattlesnake',
 "ord's kangaroo rat",
 'rattus species',
 'rodent',
 'sagebrush vole',
 'shrew-mole',
 'sorex species',
 'south-western jumping mouse',
 'southwestern speckled rattlesnake',
 'western diamondback rattlesnake',
 'western rattlesnake',
 'woodrat or rat or mouse species',
 'woodrat or rat species']

In [11]:
for cat in relevant_categories: 
    df_filtered = df.loc[df["category"].str.strip().str.lower().isin([cat,]), :]
    print(cat, ", ",   len(df_filtered))

arvicolinae subfamily ,  124342
brown rat ,  39
bushy-tailed woodrat ,  146
cactus mouse ,  76
california kangaroo rat ,  19
california vole ,  4437
desert woodrat ,  70
dusky-footed woodrat ,  3093
house mouse ,  371
house rat ,  320
mojave rattlesnake ,  50
mouse species ,  252986
muridae family ,  808
muskrat ,  10
mustelinae subfamily ,  165
north american deermouse ,  47
northern grasshopper mouse ,  7
northern pacific rattlesnake ,  65
ord's kangaroo rat ,  102
rattus species ,  19642
rodent ,  112248
sagebrush vole ,  55
shrew-mole ,  33
sorex species ,  732
south-western jumping mouse ,  46
southwestern speckled rattlesnake ,  71
western diamondback rattlesnake ,  6
western rattlesnake ,  577
woodrat or rat or mouse species ,  11846
woodrat or rat species ,  9797


In [ ]:
used_categories = [
    "brown rat",
    "house mouse", 
    "house rat", 
    # "muridae family",
    # "mouse species",
    # "rattus species", 
    # "rodent", 
    "shrew-mole", 
    "sorex species", 
]

species_names = [
    "Rattus norvegicus", 
    "Mus musculus", 
    "Rattus rattus", 
    # "Muridae", 
    # "Rattus sp.", 
    # "Rodentia",
    "Soricidae", 
    "Sorex sp."
]


for cat, sp in zip(used_categories, species_names): 
    df_filtered = df.loc[df["category"].str.strip().str.lower().isin([cat,]), :]
    print(cat, ", ",   len(df_filtered))

brown rat ,  39
house mouse ,  371
house rat ,  320
muridae family ,  808
mouse species ,  252986
rattus species ,  19642
rodent ,  112248
shrew-mole ,  33


In [22]:
df_filtered = df.loc[df["category"].str.strip().str.lower().isin(used_categories), :]
df_filtered

,image_id,id_ann,category_id,id_img,location,file_name,datetime,width,height,seq_id,seq_num_frames,frame_num,category,wi_taxon_id,class,order,family,genus,species
0,ceba8754-f8a6-4fcf-a9c8-93ac4ca77ad8,ceba8754-f8a6-4fcf-a9c8-93ac4ca77ad8_ann_00,0,ceba8754-f8a6-4fcf-a9c8-93ac4ca77ad8,2002875_sitea1-hp,2002875_cemap-small-animals_exclude-identify/i...,2020-06-21T03:50:00,2048,1440,location_2002875_sitea1-hp_sequence_index_00006,15,0,mouse species,adf63c9e-18c0-451a-a78f-4c29864bc1f5,mammalia,rodentia,NaN,NaN,NaN
2,4b99e0b0-3dbf-471b-bdba-ac021833a067,4b99e0b0-3dbf-471b-bdba-ac021833a067_ann_00,0,4b99e0b0-3dbf-471b-bdba-ac021833a067,2002875_sitea1-hp,2002875_cemap-small-animals_exclude-identify/i...,2020-06-21T03:30:00,2048,1440,location_2002875_sitea1-hp_sequence_index_00005,22,8,mouse species,adf63c9e-18c0-451a-a78f-4c29864bc1f5,mammalia,rodentia,NaN,NaN,NaN
4,2bf07b92-4aac-4ced-92cd-32cce6c4dcba,2bf07b92-4aac-4ced-92cd-32cce6c4dcba_ann_00,0,2bf07b92-4aac-4ced-92cd-32cce6c4dcba,2002875_sitea1-hp,2002875_cemap-small-animals_exclude-identify/i...,2020-06-21T03:22:00,2048,1440,location_2002875_sitea1-hp_sequence_index_00003,23,19,mouse species,adf63c9e-18c0-451a-a78f-4c29864bc1f5,mammalia,rodentia,NaN,NaN,NaN
5,70cfbaf0-897a-4db0-9f3d-bb605b5c0566,70cfbaf0-897a-4db0-9f3d-bb605b5c0566_ann_00,0,70cfbaf0-897a-4db0-9f3d-bb605b5c0566,2002875_sitea1-hp,2002875_cemap-small-animals_exclude-identify/i...,2020-06-21T04:03:00,2048,1440,location_2002875_sitea1-hp_sequence_index_00007,45,28,mouse species,adf63c9e-18c0-451a-a78f-4c29864bc1f5,mammalia,rodentia,NaN,NaN,NaN
7,eeb33521-3f68-4aa8-900f-eefcdd171e5b,eeb33521-3f68-4aa8-900f-eefcdd171e5b_ann_00,0,eeb33521-3f68-4aa8-900f-eefcdd171e5b,2002875_sitea1-hp,2002875_cemap-small-animals_exclude-identify/i...,2020-06-22T02:00:00,2048,1440,location_2002875_sitea1-hp_sequence_index_00017,8,1,mouse species,adf63c9e-18c0-451a-a78f-4c29864bc1f5,mammalia,rodentia,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2171689,9cb67d1b-731e-4cff-9781-bbd54872d991,9cb67d1b-731e-4cff-9781-bbd54872d991_ann_00,4,9cb67d1b-731e-4cff-9781-bbd54872d991,2006663_sa-29651b,2006663_mojave-small-animals_exclude-identify/...,2023-08-05T02:30:37,2048,1440,location_2006663_sa-29651b_sequence_index_01503,3,0,rodent,90d950db-2106-4bd9-a4c1-777604c3eada,mammalia,rodentia,NaN,NaN,NaN
2171690,5829773b-fb1b-4b75-9637-10ec29aef9b3,5829773b-fb1b-4b75-9637-10ec29aef9b3_ann_00,4,5829773b-fb1b-4b75-9637-10ec29aef9b3,2006663_sa-29651b,2006663_mojave-small-animals_exclude-identify/...,2023-08-05T02:30:38,2048,1440,location_2006663_sa-29651b_sequence_index_01503,3,2,rodent,90d950db-2106-4bd9-a4c1-777604c3eada,mammalia,rodentia,NaN,NaN,NaN
2171781,8004ff71-38b9-4ad8-866d-6197dccb7bd4,8004ff71-38b9-4ad8-866d-6197dccb7bd4_ann_00,4,8004ff71-38b9-4ad8-866d-6197dccb7bd4,2006663_sa-29651b,2006663_mojave-small-animals_exclude-identify/...,2023-08-06T21:48:04,2048,1440,location_2006663_sa-29651b_sequence_index_01520,6,0,rodent,90d950db-2106-4bd9-a4c1-777604c3eada,mammalia,rodentia,NaN,NaN,NaN
2171782,570659db-78c7-456b-a928-e14a21fbec36,570659db-78c7-456b-a928-e14a21fbec36_ann_00,4,570659db-78c7-456b-a928-e14a21fbec36,2006663_sa-29651b,2006663_mojave-small-animals_exclude-identify/...,2023-08-06T21:48:04,2048,1440,location_2006663_sa-29651b_sequence_index_01520,6,1,rodent,90d950db-2106-4bd9-a4c1-777604c3eada,mammalia,rodentia,NaN,NaN,NaN


this dataset has sevearl interesting annotations which can be used, in particular shrews (shrew-mole, sorex) and house- and brown rat, and house mouse Snakes might also be interesting. Checking the providence is helpful, because everything from inaturalist we already have covered. 

In particular, various snakes seem to be useful 

# download metadata for ohio small animals dataset 

In [49]:
url = "https://storage.googleapis.com/public-datasets-lila/osu-small-animals/osu-small-animals.json.zip"

name = "ohio_small_animals"
dest = "/home/hmack/Development/smartrodent_experiments/datasets"
load_metadata(url, name, out=dest)


In [74]:
import json
import pandas as pd 
metadata_path = "/home/hmack/Development/smartrodent_experiments/datasets/ohio_small_animals/osu-small-animals.json"

with open(metadata_path) as f:
    d = json.load(f)          # ~10-15 GB peak RAM; you have 42 GB free

images = pd.DataFrame(d["images"])        # id, file_name, location, datetime, dataset, seq_id, frame_num, seq_num_frames
annots = pd.DataFrame(d["annotations"])   # id, image_id, category_id
cats   = pd.DataFrame(d["categories"])    # id, name

df = annots.merge(images, left_on="image_id", right_on="id", suffixes=("_ann", "_img")).merge(cats.rename(columns={"id": "category_id", "name": "category"}), on="category_id")


sorted(df.category.unique())

['american_bullfrog',
 'american_mink',
 'american_toad',
 'brown_rat',
 "butler's_gartersnake",
 'common_five-lined_skink',
 'common_yellowthroat',
 "dekay's_brownsnake",
 'eastern_bluebird',
 'eastern_chipmunk',
 'eastern_cottontail',
 'eastern_gartersnake',
 'eastern_hog-nosed_snake',
 'eastern_massasauga',
 'eastern_milksnake',
 'eastern_racer_snake',
 'eastern_ribbonsnake',
 'empty',
 'gray_catbird',
 'gray_ratsnake',
 'green_frog',
 'indigo_bunting',
 'invertebrate',
 "kirtland's_snake",
 'long-tailed_weasel',
 'masked_shrew',
 'meadow_jumping_mouse',
 'meadow_vole',
 'n._short-tailed_shrew',
 'northern_house_wren',
 'northern_leopard_frog',
 'northern_watersnake',
 'painted_turtle',
 'plains_gartersnake',
 'raccoon',
 'red-bellied_snake',
 'smooth_greensnake',
 'snapping_turtle',
 'song_sparrow',
 'sora',
 'star-nosed_mole',
 'striped_skunk',
 'virginia_opossum',
 'white-footed_mouse',
 'woodchuck',
 'woodland_jumping_mouse']

In [75]:
sorted(list(filter(lambda x: any(f.lower() in x.lower() for f in ["rat", "mouse", "vole", "shrew", "murid", "rodent", "sorex", "crocidura", "apodemus", "Myodes", "Arvicol", "mus", "rattus"]), df.category.unique())))

['brown_rat',
 'gray_ratsnake',
 'invertebrate',
 'masked_shrew',
 'meadow_jumping_mouse',
 'meadow_vole',
 'n._short-tailed_shrew',
 'white-footed_mouse',
 'woodland_jumping_mouse']

some labels seem useful, in particular some snakes seem to be there. 

## download labels for swg camera trap data 

In [78]:
url = "https://storage.googleapis.com/public-datasets-lila/swg-camera-traps/swg_camera_traps.zip"

name = "swg_camera_traps_2018-2020"
dest = "/home/hmack/Development/smartrodent_experiments/datasets"
load_metadata(url, name, out=dest)

In [79]:
import json
import pandas as pd 
metadata_path = "/home/hmack/Development/smartrodent_experiments/datasets/swg_camera_traps_2018-2020/swg_camera_traps.json"

with open(metadata_path) as f:
    d = json.load(f)          # ~10-15 GB peak RAM; you have 42 GB free

images = pd.DataFrame(d["images"])        # id, file_name, location, datetime, dataset, seq_id, frame_num, seq_num_frames
annots = pd.DataFrame(d["annotations"])   # id, image_id, category_id
cats   = pd.DataFrame(d["categories"])    # id, name

df = annots.merge(images, left_on="image_id", right_on="id", suffixes=("_ann", "_img")).merge(cats.rename(columns={"id": "category_id", "name": "category"}), on="category_id")


sorted(df.category.unique())

['annamite_striped_rabbit',
 'asian_fairy_bluebird',
 'asiatic_black_bear',
 'asiatic_brush_tailed_porcupine',
 'assam_or_rhesus_macaque',
 'back_striped_weasel',
 'bamboo_rat',
 'bar_bellied_pitta',
 'barred_cuckoo_dove',
 'bengal_monitor',
 'binturong',
 'black_giant_squirrel',
 'black_throated_laughingthrush',
 'blue_fronted_robin',
 'blue_pitta',
 'blue_whistling_thrush',
 'blurred',
 'brown_wood_owl',
 'chevrotain',
 'chinese serow',
 'chinese_serow',
 'common_green_magpie',
 'common_palm_civet',
 'crab_eating_mongoose',
 'crested_argus',
 'crested_serpent_eagle',
 'domestic_dog',
 'emerald_dove',
 'empty',
 'eurasian_wild_pig',
 'eurasian_woodcock',
 'eyebrowed_thrush',
 'ferret_badger',
 'flying_squirrel',
 'forktail_sp',
 'francois_langur',
 'giant_flying_squirrel',
 'gibbon',
 'golden_babbler',
 'golden_cat',
 'gray_laughingthrush',
 'grey_headed_canary_flycatcher',
 'grey_peacock_pheasant',
 'hatinh_langur',
 'hog_badger',
 'human',
 'ignore',
 'indochinese_green_magpie',
 'i

In [80]:
sorted(list(filter(lambda x: any(f.lower() in x.lower() for f in ["rat", "mouse", "vole", "shrew", "murid", "rodent", "sorex", "crocidura", "apodemus", "Myodes", "Arvicol", "mus", "rattus"]), df.category.unique())))

['bamboo_rat', 'invertebrate', 'northern_treeshrew', 'unidentified_murid']

doesn´t seem particularly interesting, unless we want more 'murid' images

# download metadata for Idaho camera trap dataset 

In [51]:
url = "https://storage.googleapis.com/public-datasets-lila/idaho-camera-traps/idaho-camera-traps.json.zip"

name = "idaho_camera_trap_data"
dest = "/home/hmack/Development/smartrodent_experiments/datasets"
load_metadata(url, name, out=dest)

In [81]:
import json
import pandas as pd 
metadata_path = "/home/hmack/Development/smartrodent_experiments/datasets/idaho_camera_trap_data/idaho-camera-traps.json"

with open(metadata_path) as f:
    d = json.load(f)          # ~10-15 GB peak RAM; you have 42 GB free

images = pd.DataFrame(d["images"])        # id, file_name, location, datetime, dataset, seq_id, frame_num, seq_num_frames
annots = pd.DataFrame(d["annotations"])   # id, image_id, category_id
cats   = pd.DataFrame(d["categories"])    # id, name

df = annots.merge(images, left_on="image_id", right_on="id", suffixes=("_ann", "_img")).merge(cats.rename(columns={"id": "category_id", "name": "category"}), on="category_id")


sorted(df.category.unique())

['badger',
 'bat',
 'bear',
 'bighorn sheep',
 'bird',
 'bluebird',
 'bobcat',
 'cattle',
 "clark's nutcracker",
 'coyote',
 'crow',
 'deer',
 'domestic dog',
 'elk',
 'empty',
 'ermine',
 'fisher',
 'foggy lens',
 'foggy weather',
 'fox',
 'grouse',
 'hawk',
 'horse',
 'human',
 'hummingbird',
 'lagomorph',
 'lens obscured',
 'long-tailed weasel',
 'magpie',
 'malfunction',
 'misdirected',
 'moose',
 'moth',
 'mountain lion',
 'other',
 'owl',
 'porcupine',
 'pronghorn',
 'rabbit',
 'raccoon',
 'raven',
 'red-breasted nuthatch',
 'rodent',
 'skunk',
 'snow on lens',
 'squirrel',
 'sun',
 'tilted',
 'turkey',
 'unknown',
 'unknown bird of prey',
 'unknown canid',
 'unknown cat',
 'unknown cervid',
 'unknown ungulate',
 'varied thrush',
 'vegetation obstruction',
 'vehicle',
 'warbler',
 'western tanager',
 'wolf',
 'woodpecker']

In [82]:
sorted(list(filter(lambda x: any(f.lower() in x.lower() for f in ["rat", "mouse", "vole", "shrew", "murid", "rodent", "sorex", "crocidura", "apodemus", "Myodes", "Arvicol", "mus", "rattus"]), df.category.unique())))

['rodent']

not good, unless we want various non-rodent images. 

# download metadata for North American Camera Trap dataset

In [52]:
url = "https://storage.googleapis.com/public-datasets-lila/nacti/nacti_metadata.csv.zip"

name = "north_america_camera_trap_data"
dest = "/home/hmack/Development/smartrodent_experiments/datasets"
load_metadata(url, name, out=dest)

In [25]:
import pandas as pd 
metadata_path = "/home/hmack/Development/smartrodent_experiments/datasets/north_america_camera_trap_data/nacti_metadata.csv"

df = pd.read_csv(metadata_path).dropna() # dorp the empty stuff
df

/tmp/ipykernel_337054/4089738692.py:4: DtypeWarning: Columns (0: study, 1: location, 2: genus, 3: family, 4: order, 5: class) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(metadata_path).dropna() # dorp the empty stuff


,seq_no,id,filename,study,location,width,height,category_id,name,genus,family,order,class,common_name
0,0,2010_Unit150_Ivan097_img0001.jpg,part0/sub000/2010_Unit150_Ivan097_img0001.jpg,CPW,"San Juan Mntns, Colorado",2048,1536,10,cervus elaphus,cervus,cervidae,artiodactyla,mammalia,red deer
1,1,2010_Unit150_Ivan097_img0002.jpg,part0/sub000/2010_Unit150_Ivan097_img0002.jpg,CPW,"San Juan Mntns, Colorado",2048,1536,10,cervus elaphus,cervus,cervidae,artiodactyla,mammalia,red deer
2,2,2010_Unit150_Ivan097_img0003.jpg,part0/sub000/2010_Unit150_Ivan097_img0003.jpg,CPW,"San Juan Mntns, Colorado",2048,1536,10,cervus elaphus,cervus,cervidae,artiodactyla,mammalia,red deer
3,3,2010_Unit150_Ivan097_img0004.jpg,part0/sub000/2010_Unit150_Ivan097_img0004.jpg,CPW,"San Juan Mntns, Colorado",2048,1536,10,cervus elaphus,cervus,cervidae,artiodactyla,mammalia,red deer
4,4,2010_Unit150_Ivan097_img0005.jpg,part0/sub000/2010_Unit150_Ivan097_img0005.jpg,CPW,"San Juan Mntns, Colorado",2048,1536,10,cervus elaphus,cervus,cervidae,artiodactyla,mammalia,red deer
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2912850,2912850,TAG-TC36_11_17_2015_TAG-TC36_0004950.JPG,part2/sub291/TAG-TC36_11_17_2015_TAG-TC36_0004...,TAG,"Lebec, California",2048,1536,50,otospermophilus beecheyi,otospermophilus,sciuridae,rodentia,mammalia,california ground squirrel
2912852,2912852,TAG-TC36_11_17_2015_TAG-TC36_0005837.JPG,part2/sub291/TAG-TC36_11_17_2015_TAG-TC36_0005...,TAG,"Lebec, California",2048,1536,50,otospermophilus beecheyi,otospermophilus,sciuridae,rodentia,mammalia,california ground squirrel
2912853,2912853,TAG-TC36_11_17_2015_TAG-TC36_0005838.JPG,part2/sub291/TAG-TC36_11_17_2015_TAG-TC36_0005...,TAG,"Lebec, California",2048,1536,50,otospermophilus beecheyi,otospermophilus,sciuridae,rodentia,mammalia,california ground squirrel
2912857,2912857,TAG-TC36_11_17_2015_TAG-TC36_0007118.JPG,part2/sub291/TAG-TC36_11_17_2015_TAG-TC36_0007...,TAG,"Lebec, California",2048,1536,3,bos taurus,bos,bovidae,artiodactyla,mammalia,domestic cow


In [26]:
sorted(df.name.unique())

['alces alces',
 'bos taurus',
 'callipepla californica',
 'canis familiaris',
 'canis latrans',
 'cervus canadensis',
 'cervus elaphus',
 'cyanocitta stelleri',
 'dasypus novemcinctus',
 'dendragapus obscurus',
 'didelphis virginiana',
 'equus africanus',
 'equus ferus',
 'erethizon dorsatum',
 'junco hyemalis',
 'lepus americanus',
 'lepus californicus',
 'lontra canadensis',
 'lynx rufus',
 'marmota flaviventris',
 'martes americana',
 'meleagris gallopavo',
 'meles meles',
 'mephitis mephitis',
 'mustela erminea',
 'mustela frenata',
 'odocoileus hemionus',
 'otospermophilus beecheyi',
 'perisoreus canadensis',
 'procyon lotor',
 'puma concolor',
 'sciurus carolinensis',
 'sus scrofa',
 'tamiasciurus hudsonicus',
 'troglodytes aedon',
 'unidentified chipmunk',
 'unidentified corvus',
 'unidentified deer',
 'unidentified deer mouse',
 'unidentified mouse',
 'unidentified pack rat',
 'unidentified pocket gopher',
 'unidentified rabbit',
 'urocyon cinereoargenteus',
 'ursus americanus

In [27]:
sorted(df.family.unique())

['bovidae',
 'canidae',
 'cervidae',
 'corvidae',
 'cricetidae',
 'dasypodidae',
 'didelphidae',
 'equidae',
 'erethizontidae',
 'felidae',
 'geomyidae',
 'leporidae',
 'mephitidae',
 'muridae',
 'mustelidae',
 'odotophoridae',
 'passerellidae',
 'phasianidae',
 'procyonidae',
 'sciuridae',
 'suidae',
 'troglodytidae',
 'ursidae']

In [28]:
sorted(list(filter(lambda x: any(f.lower() in x.lower() for f in ["rat", "mouse", "vole", "shrew", "murid", "rodent", "sorex", "crocidura", "apodemus", "Myodes", "Arvicol", "mus", "rattus"]), df.name.unique())))

['mustela erminea',
 'mustela frenata',
 'unidentified deer mouse',
 'unidentified mouse',
 'unidentified pack rat']

In [29]:
sorted(list(filter(lambda x: any(f.lower() in x.lower() for f in ["rat", "mouse", "vole", "shrew", "murid", "rodent", "sorex", "crocidura", "apodemus", "Myodes", "Arvicol", "mus", "rattus"]), df.common_name.unique())))

['unidentified deer mouse', 'unidentified mouse', 'unidentified pack rat']

In [30]:
sorted(list(filter(lambda x: any(f.lower() in str(x).lower() for f in ["rat", "mouse", "vole", "shrew", "murid", "rodent", "sorex", "crocidura", "apodemus", "Myodes", "Arvicol", "mus", "rattus"]), df.family.unique())))

['muridae', 'mustelidae']

seems to be of limited use for our purposes. 